In [7]:
import pandas as pd
import numpy as np

r_cols = ['user_id', 'movie_id', 'rating']
# le o arquivo de avaliacoes (u.data), separado por tab, e usa so as 3 primeiras colunas
ratings = pd.read_csv('C:/Users/zinho/OneDrive/Documentos/Lamia/Card 11 - Prática Lidando com Dados do Mundo Real (II)/Aquivos_de_C%C3%B3digo/u.data', sep='\t', names=r_cols, usecols=range(3))
ratings.head()  # mostra as primeiras linhas p conferir se leu certo

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [8]:
# agrupa por filme e calcula qtd de avaliacoes (size) e nota media (mean) de cada um
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})
movieProperties.head()

C:\Users\zinho\AppData\Local\Temp\ipykernel_72024\2505249100.py:1: FutureWarning: The provided callable <function mean at 0x000002D97E57E560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})


rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [10]:
# pega so a coluna de qtd de avaliacoes (o "size") de cada filme
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
# normaliza (min-max) p ficar entre 0 e 1, assim da p comparar popularidade numa escala igual p todos
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)))
movieNormalizedNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


In [13]:
movieDict = {}
with open('C:/Users/zinho/OneDrive/Documentos/Lamia/Card 11 - Prática Lidando com Dados do Mundo Real (II)/Aquivos_de_C%C3%B3digo/u.item') as f:  # abre arquivo com infos dos filmes (nome, generos etc)
    temp = ''
    for line in f:
        fields = line.rstrip('\n').split('|')  # cada linha eh separada por |
        movieID = int(fields[0])
        name = fields[1]
        genres = fields[5:25]  # os generos vem como flags 0/1 nessas posicoes
        genres = list(map(int, genres))  # converte cada flag de string p int
        # guarda no dict: nome, generos, popularidade normalizada e (qtd, media) de avaliacoes
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating)

In [14]:
movieDict[1]  # confere como ficou os dados do filme id 1 (Toy Story)

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)

In [15]:
from scipy import spatial
def ComputeDistance(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB)  # distancia de cosseno entre os generos
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)  # diferenca de popularidade
    return genreDistance + popularityDistance  # distancia total = soma das duas

ComputeDistance(movieDict[2], movieDict[4])  # testa a funcao c 2 filmes quaisquer

np.float64(0.8004574042309892)

In [17]:
print(movieDict[2])
print(movieDict[4])  # só olhando os dados brutos dos 2 filmes usados no teste de cima

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), size    131.000000
mean      3.206107
Name: 2, dtype: float64)
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), size    209.000000
mean      3.550239
Name: 4, dtype: float64)


In [19]:
import operator

def getNeighbors(movieID, K):
    distances = []
    for movie in movieDict:
        if (movie != movieID):  # nao compara o filme com ele msm
            dist = ComputeDistance(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))
    distances.sort(key=operator.itemgetter(1))  # ordena do mais parecido (menor distancia) p mais diferente
    neighbors = []
    for x in range(K):
        neighbors.append(distances[x][0])  # pega so os K mais proximos, os "vizinhos"
    return neighbors

K = 10
avgRating = 0
neighbors = getNeighbors(1, K)  # acha os 10 filmes mais parecidos c o Toy Story (id 1)
for neighbor in neighbors:
    avgRating += movieDict[neighbor][3]
    print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))

avgRating /= float(K)  # media das notas dos vizinhos = a previsao de nota do KNN

Liar Liar (1997) size    485.000000
mean      3.156701
Name: 294, dtype: float64
Aladdin (1992) size    219.000000
mean      3.812785
Name: 95, dtype: float64
Willy Wonka and the Chocolate Factory (1971) size    326.000000
mean      3.631902
Name: 151, dtype: float64
Monty Python and the Holy Grail (1974) size    316.000000
mean      4.066456
Name: 168, dtype: float64
Full Monty, The (1997) size    315.000000
mean      3.926984
Name: 269, dtype: float64
George of the Jungle (1997) size    162.000000
mean      2.685185
Name: 259, dtype: float64
Beavis and Butt-head Do America (1996) size    156.000000
mean      2.788462
Name: 240, dtype: float64
Birdcage, The (1996) size    293.000000
mean      3.443686
Name: 25, dtype: float64
Home Alone (1990) size    137.000000
mean      3.087591
Name: 94, dtype: float64
Aladdin and the King of Thieves (1996) size    26.000000
mean     2.846154
Name: 422, dtype: float64


In [20]:
avgRating  # nota prevista p o filme, baseado na media dos vizinhos (essa eh a predicao do KNN)

size    243.500000
mean      3.344591
Name: 294, dtype: float64

In [21]:
movieDict[1]  # so revendo os dados do filme original dnv

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)